# Modelo economico EVA Valle - Exploracion 1
Objetivo: convertir toneladas en pesos (PIB agro municipal) usando
precios de referencia v0 (supuesto declarado) + precios de tierra (datos.gov.co).
Ejecuta las celdas en orden.

In [1]:
import sys
from pathlib import Path
import pandas as pd

# Ubica la raiz del proyecto sin depender de donde se lanzo Jupyter
root = Path.cwd()
while root != root.parent and not (root / "app.py").exists():
    root = root.parent
if not (root / "app.py").exists():
    root = Path("C:/Users/Usuario/Documents/eva-valle-v3.0")
sys.path.insert(0, str(root))

from config.settings import settings

df = pd.read_csv(settings.DATA_MODEL_PATH / 'eva_agricola_valle_modelo_conceptual.csv',
                 low_memory=False)
print('EVA:', df.shape, '| raiz:', root)
df.head()

EVA: (10589, 19) | raiz: C:\Users\Usuario\Documentos\eva-valle-v3.0


,codigo_dane_departamento,departamento,codigo_dane_municipio,municipio,desagregacion_cultivo,cultivo,ciclo_del_cultivo,grupo_cultivo,subgrupo,ano,periodo,area_sembrada_ha,area_cosechada_ha,produccion_t,rendimiento_t_ha,nombre_cientifico_del_cultivo,codigo_del_cultivo,estado_fisico_del_cultivo,id_registro
0,76,Valle del Cauca,76001,Santiago de Cali,Acelga,Acelga,Transitorio,Hortalizas,Hortalizas de hoja,2019,2019A,3.0,3.0,12.0,4.0,Beta vulgaris var. cicla,1050100,En fresco,EVA-2019-2019A-76001-00001
1,76,Valle del Cauca,76001,Santiago de Cali,Acelga,Acelga,Transitorio,Hortalizas,Hortalizas de hoja,2019,2019B,4.0,4.0,16.0,4.0,Beta vulgaris var. cicla,1050100,En fresco,EVA-2019-2019B-76001-00002
2,76,Valle del Cauca,76001,Santiago de Cali,Acelga,Acelga,Transitorio,Hortalizas,Hortalizas de hoja,2020,2020A,3.0,3.0,12.0,4.0,Beta vulgaris var. cicla,1050100,En fresco,EVA-2020-2020A-76001-00003
3,76,Valle del Cauca,76001,Santiago de Cali,Acelga,Acelga,Transitorio,Hortalizas,Hortalizas de hoja,2020,2020B,2.5,2.5,10.0,4.0,Beta vulgaris var. cicla,1050100,En fresco,EVA-2020-2020B-76001-00004
4,76,Valle del Cauca,76001,Santiago de Cali,Acelga,Acelga,Transitorio,Hortalizas,Hortalizas de hoja,2021,2021A,3.0,3.0,12.0,4.0,Beta vulgaris var. cicla,1050100,En fresco,EVA-2021-2021A-76001-00005


## 1) Precios comerciales de la tierra rural (UPRA, datos.gov.co)
Descarga directa del dataset abierto rttb-pk7n.

In [2]:
URL_TIERRA = 'https://www.datos.gov.co/resource/rttb-pk7n.csv'
tierra = None
try:
    tierra = pd.read_csv(URL_TIERRA, nrows=20000)
    print('[OK] tierra:', tierra.shape)
    tierra.head()
except Exception as e:
    print('[AVISO] sin descarga de tierra:', e)

[OK] tierra: (1000, 10)


## 2) Precios de referencia v0 (supuesto metodologico)
Tabla editable: ajusta valores y re-ejecuta. La celda reporta cobertura.

In [3]:
PRECIOS_REF = {  # COP/t, v0 (validar con Primer Mercado UPRA)
    "Caña": 160000, "Caña de azúcar": 160000,
    "Plátano": 1200000, "Banano": 1200000,
    "Naranja": 700000, "Mandarina": 900000, "Tomate": 1500000,
    "Piña": 900000, "Maracuyá": 3500000, "Papaya": 1000000,
    "Café": 2800000, "Aguacate": 2500000, "Yuca": 900000,
    "Maíz": 1100000, "Cacao": 12000000, "Guanábana": 2000000,
    "Guayaba": 800000,
}
precios = pd.Series(PRECIOS_REF, name='precio_t')
cub = df['cultivo'].map(precios).notna()
print(f'Cobertura filas: {cub.mean():.1%} | tonelaje cubierto: '
      f"{df.loc[cub, 'produccion_t'].sum() / df['produccion_t'].sum():.1%}")
if cub.mean() < 0.8:
    print('Cultivos sin precio:', sorted(set(df.loc[~cub, 'cultivo'].unique())))

Cobertura filas: 51.3% | tonelaje cubierto: 99.1%
Cultivos sin precio: ['Acelga', 'Ahuyama', 'Ají', 'Algodón', 'Arazá', 'Arracacha', 'Arroz', 'Arveja', 'Berenjena', 'Borojó', 'Brevo', 'Brócoli', 'Calabacín, calabaza', 'Cebolla de bulbo', 'Cebolla de rama', 'Chontaduro', 'Cilantro', 'Cimarrón', 'Coco', 'Coliflor', 'Curuba', 'Cúrcuma o azafrán', 'Durazno o albaricoque', 'Espinaca', 'Fresa', 'Frijol', 'Granadilla', 'Gulupa o cholupa', 'Habichuela', 'Lechuga', 'Lima', 'Limón', 'Lulo', 'Macadamia', 'Malanga, achín, yota, papa china, bore', 'Mango', 'Melón', 'Mora', 'Otras hortalizas', 'Otros cítricos', 'Papa', 'Patilla', 'Pepino Cohombro', 'Perejil', 'Pimentón', 'Pitahaya', 'Plantas aromáticas', 'Remolacha', 'Repollo', 'Romero', 'Rábano', 'Sacha inchi', 'Sorgo', 'Soya', 'Stevia', 'Sábila', 'Tomate de árbol', 'Té', 'Uchuva', 'Uva', 'Zanahoria', 'Zapote']


## 3) PIB agro municipal 2025: ranking en PESOS vs ranking en TONELADAS
Aqui aparece la sorpresa economica: el ranking en pesos NO es el ranking en toneladas.

In [4]:
d = df.copy()
d['valor'] = d['produccion_t'] * d['cultivo'].map(precios)
g25 = d[d.ano == 2025].groupby('municipio')['valor'].sum()
t25 = d[d.ano == 2025].groupby('municipio')['produccion_t'].sum()
comp = pd.DataFrame({'PIB_agro_2025_M_COP': (g25 / 1e6).round(0),
                     'ton_2025': t25}).dropna()
comp['rank_pesos'] = comp['PIB_agro_2025_M_COP'].rank(ascending=False).astype(int)
comp['rank_ton'] = comp['ton_2025'].rank(ascending=False).astype(int)
comp['salto_rank'] = comp['rank_ton'] - comp['rank_pesos']
comp.sort_values('PIB_agro_2025_M_COP', ascending=False).head(10)

,PIB_agro_2025_M_COP,ton_2025,rank_pesos,rank_ton,salto_rank
municipio,,,,,
Palmira,783670.0,4789231.10,1,1,0
Candelaria,512277.0,3189450.36,2,2,0
Zarzal,342985.0,2129868.41,3,3,0
El Cerrito,332233.0,2068976.27,4,4,0
Guacarí,291499.0,1724737.50,5,5,0
Florida,287081.0,1630501.61,6,6,0
Bugalagrande,282777.0,1523305.43,7,7,0
Sevilla,211171.0,187469.12,8,26,18
Pradera,208054.0,1274310.21,9,8,-1


## 4) PIB agro departamental anual y CAGR en pesos

In [5]:
serie = d.groupby('ano')['valor'].sum() / 1e9
print('PIB agro departamental (miles de M COP):')
print(serie.round(1))
n = serie.index[-1] - serie.index[0]
cagr_pesos = ((serie.iloc[-1] / serie.iloc[0]) ** (1 / n) - 1) * 100
print(f'CAGR en pesos 2019-2025: {cagr_pesos:+.1f}%')

PIB agro departamental (miles de M COP):
ano
2019    4858.8
2020    4863.2
2021    5128.2
2022    5351.6
2023    5501.8
2024    5786.7
2025    5931.9
Name: valor, dtype: float64
CAGR en pesos 2019-2025: +3.4%


## 5) Exportar para la app

In [6]:
out = Path('data'); out.mkdir(exist_ok=True)
comp.to_csv(out / 'economia_municipal_2025.csv')
serie.rename('pib_miles_M').to_csv(out / 'pib_agro_anual.csv')
print('[OK] exportado a data/economia_municipal_2025.csv y data/pib_agro_anual.csv')

[OK] exportado a data/economia_municipal_2025.csv y data/pib_agro_anual.csv
